# K4TQ scaled: K4TQ-G "Ginna" (≤ 12.5 kg) against Freddy (K4TQ-B "Bertha", ≤ 24 kg, is parked: too heavy)

| design | what it is |
|---|---|
| **K4TQ-F1 "Freddy"** | the reference: 1.2 m fuselage, 1.4 m span, MN501-S KV300 + 16x10, 6S3P, front NACA 4415 / rear NACA 2415 |
| **K4TQ-G "Ginna"** | Freddy × 1.35, take-off mass ≤ 12.5 kg: U8 II KV170 + 20x10, 12S3P |
| ~~K4TQ-B "Bertha"~~ | parked (too heavy): Freddy × 1.85 at ≤ 24 kg, U12 II KV100 + 28x17, 12S4P, 5.15 kg payload — still in `k4tq.VARIANTS["B"]`; add `"B"` to `DESIGNS` below to see it |

Every length scales (stations stay fractions of the fuselage; the cargo cartridge grows with it); the hardware goes up a
class (12S packs of the same Molicel P45B cells, 12S ESCs, large carbon props on folding hubs, the smallest wing-root
hardware whose actuator, bearings and locking pin hold); the spar from the root loads. No CAD of the scaled designs yet:
the made parts' estimates are corrected by the factors Freddy's CAD found (`k4tq.MADE_CALIBRATION`).

**The sizing rule** (`k4tq_scaled.size`): the most payload on the standard mission R3 (10 km radius, 2 min hover, 20 %
reserve, 8 m/s wind) within the mass limit, with the hover rules (T/W ≥ 1.6 on a sagged pack, most loaded motor ≤ 75 %)
and **Freddy's stall speed (≤ 18 m/s)** — the same transition corridor and wing loading. The stall cap is what sets the
scale: a smaller airframe would carry more payload at the limit mass but stall faster. Labels: ESTIMATE / CALC (MOCK BEMT
under `DEBUG = True`); every new catalogue part is "to verify".

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import k4tq, k4tq_sizing as sz, k4tq_scaled as ks
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
DESIGNS = ("F1", "G")                 # + "B" (Bertha, parked)
for d in DESIGNS:
    print(f"K4TQ-{d:3s} {k4tq.params(d)['title']}")

## 1. Side by side (about two minutes)

In [ ]:
df = ks.summary(designs=DESIGNS[1:])
df.T

## 2. What payload can each take — and what limits it

In [ ]:
lim = pd.DataFrame({f"K4TQ-{d} ({k4tq.NICKNAME[d]})": ks.payload_limits(d) for d in DESIGNS}).T
lim

## 3. Mass breakdown

In [ ]:
def groups(d):
    v = sz.vehicle(d)
    g = {}
    for i in v["items"]:
        n = i["item"]
        k = ("wing roots" if n.startswith("wing root") else "wing panels" if n.startswith("wing panel") else "spars" if n.startswith("spar")
             else "motors" if n.startswith("motor") else "propellers" if n.startswith("propeller") else "ESCs" if n.startswith("ESC")
             else "fuselage" if n.startswith("fuselage") else i["group"])
        g[k] = g.get(k, 0.0) + i["mass_kg"]
    return pd.Series(g)
pd.DataFrame({d: groups(d) for d in DESIGNS}).round(2).sort_values(DESIGNS[-1], ascending=False)

## 4. The scaled geometry (the CAD built from the scaled parameters)

In [ ]:
for d in DESIGNS[1:]:
    v = sz.vehicle(d)
    import k4tq_cad
    png = k4tq_cad.render(Path(k4tq.__file__).parent / "data" / "k4tq" / "figures" / f"k4tq_{d}.png", v["p"], info={"mass": v["mass"]["mass_kg"]}, items=v["items"])
    display(Image(str(png)))

## 5. Re-running the sizing (slow: ~10 min per design under the mocks)

```python
r = ks.size("G")                      # or ks.size("B", scales=(1.8, 1.85, 1.9))
r["table"], r["best"]
```

## Reading it

- **Payload**: Freddy 1 kg as designed (hover margin caps it at ≈1.25 kg; the energy would carry ≈2.5 kg);
  **Ginna ≈2.6 kg** (the 12.5 kg limit caps it — the energy would carry ≈6 kg, the hover ≈3.8 kg);
  (Bertha, parked: ≈5.2 kg at 24 kg).
- **Why not more**: the airframe's empty mass grows faster than its size (roots, spar and shells ∝ s²…s³), and the stall
  cap stops it shrinking. A 2× K4TQ would be ≈35 kg.
- **Range**: Ginna 59 km still-air (as Freddy), R3 radius ≈14 km.
- **Hover power**: ≈2.0 kW (Ginna) — downwash, noise and the landing pad grow with it.
- **Open**: the CAD of the scaled roots (the render's interference checks flag the gear/frame and cartridge envelopes
  that were sized for Freddy), the transition corridor at the new scale, real datasheets for the 12S motors and the
  Mejzlik propellers, the operation category (Bertha needs an operational authorisation in the specific category even
  under 25 kg — one reason it is parked; Ginna fits C-class rules more easily).